# StyleTTS 2 Demo (LibriTTS)

Before you run the following cells, please make sure you have downloaded [reference_audio.zip](https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/reference_audio.zip) and unzipped it under the `demo` folder.

### Utils

In [54]:
import torch
torch.manual_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import random
random.seed(0)

import numpy as np
np.random.seed(0)

In [46]:
%cd ..

# scp  user@46.18.108.33:/home/user/voice/StyleTTS2/Models/indic_voices/epoch_2nd_00014.pth /home/cmi_10101/Documents/voice/Hindi/StyleTTS2/Models/indicvoices


/home/user/voice


In [55]:
# load packages
import time
import random
import yaml
import scipy.signal
from munch import Munch
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import torchaudio
import librosa
from nltk.tokenize import word_tokenize
import sys 
sys.path.append('/home/user/voice/StyleTTS2')
from models import *
from utils import *
from text_utils import TextCleaner
textclenaer = TextCleaner()

%matplotlib inline

177


In [56]:
to_mel = torchaudio.transforms.MelSpectrogram(
    n_mels=80, n_fft=2048, win_length=1200, hop_length=300)
mean, std = -4, 4

def length_to_mask(lengths):
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    mask = torch.gt(mask+1, lengths.unsqueeze(1))
    return mask

def preprocess(wave):
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor = to_mel(wave_tensor)
    mel_tensor = (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std
    return mel_tensor

def compute_style(path):
    wave, sr = librosa.load(path, sr=24000)
    audio, index = librosa.effects.trim(wave, top_db=30)
    if sr != 24000:
        audio = librosa.resample(audio, sr, 24000)
    mel_tensor = preprocess(audio).to(device)

    with torch.no_grad():
        ref_s = model.style_encoder(mel_tensor.unsqueeze(1))
        ref_p = model.predictor_encoder(mel_tensor.unsqueeze(1))

    return torch.cat([ref_s, ref_p], dim=1)

In [57]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

### Load models

In [58]:
# load phonemizer
import phonemizer
global_phonemizer = phonemizer.backend.EspeakBackend(language='hi', preserve_punctuation=True,  with_stress=True)
global_phonemizer_en = phonemizer.backend.EspeakBackend(language='en-us', preserve_punctuation=True,  with_stress=True)

In [59]:
config = yaml.safe_load(open("/home/user/voice/StyleTTS2/Models/telecmi/config_ft.yml"))

# load pretrained ASR model
ASR_config = config.get('ASR_config', False)
ASR_path = config.get('ASR_path', False)
text_aligner = load_ASR_models(ASR_path, ASR_config)

# load pretrained F0 model
F0_path = config.get('F0_path', False)
pitch_extractor = load_F0_models(F0_path)

# load BERT model
from Utils.PLBERT.util import load_plbert
BERT_path = config.get('PLBERT_dir', False)
plbert = load_plbert(BERT_path)

In [60]:
model_params = recursive_munch(config['model_params'])
model = build_model(model_params, text_aligner, pitch_extractor, plbert)
_ = [model[key].eval() for key in model]
_ = [model[key].to(device) for key in model]

/home/user/anaconda3/envs/styletts/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [61]:
params_whole = torch.load("/home/user/voice/StyleTTS2/Models/telecmi/epoch_2nd_00009.pth", map_location='cpu')
params = params_whole['net']

In [62]:
for key in model:
    if key in params:
        print('%s loaded' % key)
        try:
            model[key].load_state_dict(params[key])
        except:
            from collections import OrderedDict
            state_dict = params[key]
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:] # remove `module.`
                new_state_dict[name] = v
            # load params
            model[key].load_state_dict(new_state_dict, strict=False)
#             except:
#                 _load(params[key], model[key])
_ = [model[key].eval() for key in model]

bert loaded
bert_encoder loaded
predictor loaded
decoder loaded
text_encoder loaded
predictor_encoder loaded
style_encoder loaded
diffusion loaded
text_aligner loaded
pitch_extractor loaded
mpd loaded
msd loaded
wd loaded


In [63]:
from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

In [64]:
sampler = DiffusionSampler(
    model.diffusion.diffusion,
    sampler=ADPM2Sampler(),
    sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0), # empirical parameters
    clamp=False
)

In [65]:
# from phonemizer import phonemize
# from phonemizer.separator import Separator

# def clean_phonemize(
#     text: str,
#     language: str = "en-us",
#     backend: str = "espeak",
#     njobs: int = 4
# ) -> str:
#     """
#     Phonemize a mixed Hindi-English string into a plain IPA-like string,
#     removing any language-switch tags like (en) or (hi).

#     Args:
#         text (str): Input sentence (e.g., "CRM_SALES AI असिस्टेड कॉल लॉग Received.")
#         language (str): Main language code for phonemizer (default 'hi').
#         backend (str): Phonemizer backend to use (default 'espeak').
#         njobs (int): Number of parallel jobs (default 4).

#     Returns:
#         str: Phoneme transcription without language-switch flags.
#     """
#     # 1. Configure separators: no delimiter for phones, a single space for words
#     separator = Separator(phone="", word=" ", syllable="")

#     # 2. Invoke high-level phonemize with remove-flags policy
#     phoneme_list = phonemize(
#         [text],
#         language=language,
#         backend=backend,
#         separator=separator,
#         strip=True,                   # trim leading/trailing whitespace
#         preserve_punctuation=True,    # keep punctuation intact
#         njobs=njobs,
#         language_switch="remove-flags"  # strip out (en)/(hi) tags :contentReference[oaicite:0]{index=0}
#     )

#     # 3. Return the first (and only) phoneme string
#     return phoneme_list

## update language mixed clean phonemize function #####


# from phonemizer import phonemize
# from phonemizer.separator import Separator
# from phonemizer.backend import EspeakBackend
# import re

# def preprocess_text(text):
#     """Fix typos and standardize words."""
#     text = text.replace("web side", "website")  # Correct typo
#     text = text.replace("Sir", "सर")  # Use Hindi "sar" (optional)
#     return text

# def segment_text_by_script(text):
#     """Split text into Hindi and English segments."""
#     devanagari_pattern = r'[\u0900-\u097F]+'
#     latin_pattern = r'[a-zA-Z0-9]+'
    
#     segments = []
#     current_lang = None
#     current_segment = []
    
#     for word in text.split():
#         if re.match(devanagari_pattern, word):
#             lang = 'hi'
#         elif re.match(latin_pattern, word):
#             # Use 'en-gb' instead of 'en-IN' as it's supported by phonemizer
#             # and closer to Indian English than American English
#             lang = 'en-us'
#         else:
#             lang = current_lang if current_lang else 'hi'
        
#         if lang != current_lang and current_segment:
#             segments.append((' '.join(current_segment), current_lang))
#             current_segment = []
#         current_segment.append(word)
#         current_lang = lang
    
#     if current_segment:
#         segments.append((' '.join(current_segment), current_lang))
    
#     return segments

# def indian_english_post_process(phonemes):
#     """Apply Indian English specific pronunciation adjustments"""
#     # Add any Indian English specific pronunciation patterns here
#     # Examples might include:
#     phonemes = re.sub(r'əʊ', 'oː', phonemes)  # Change 'phone' vowel to more Indian style
#     phonemes = re.sub(r'eɪ', 'eː', phonemes)  # Change 'face' vowel
#     phonemes = re.sub(r'r\b', 'r', phonemes)  # Ensure rhotic r at word endings
    
#     return phonemes

# def check_language_support():
#     """Check if Indian English is supported in this espeak installation."""
#     backend = EspeakBackend('en')
#     available_languages = backend.supported_languages()
    
#     # Check for Indian English directly
#     has_indian_english = 'en-in' in available_languages
    
#     # Print available languages for debugging
#     print(f"Available espeak languages: {', '.join(sorted(available_languages))}")
    
#     if has_indian_english:
#         print("Indian English (en-in) is available in your espeak installation!")
#         return 'en-in'
#     else:
#         print("Indian English is not available, using en-gb as fallback")
#         return 'en-gb'

# def clean_phonemize(text):
#     """Phonemize mixed Hindi-English text with preprocessing and post-processing."""
#     # Preprocess
#     text = preprocess_text(text)
    
#     # Segment
#     segments = segment_text_by_script(text)
    
#     # Phonemize
#     separator = Separator(phone="", word=" ", syllable="")
#     phoneme_parts = []
    
#     for segment, lang in segments:
#         # Use the espeak backend through the phonemize function
#         phonemes = phonemize(
#             [segment],
#             language=lang,
#             backend="espeak",
#             separator=separator,
#             strip=True,
#             preserve_punctuation=True,
#             njobs=1,  # Single process is more stable with espeak
#             language_switch="remove-flags"
#         )[0]
        
#         # Apply Indian English post-processing if it's English
#         if lang == 'en-us':
#             phonemes = indian_english_post_process(phonemes)
        
#         phoneme_parts.append(phonemes)
    
#     # Combine
#     combined_phonemes = ' '.join(phoneme_parts)
    
#     # General post-processing
#     combined_phonemes = combined_phonemes.replace('sɜː', 'sər')  # Fix "Sir"
#     combined_phonemes = combined_phonemes.replace('wɛb saɪd', 'wɛbsaɪt')  # Fix "web side"
    
#     return combined_phonemes.split()

In [66]:
from phonemizer import phonemize
from phonemizer.separator import Separator
import re

def preprocess_text(text):
    text = text.replace("web side", "website")
    text = text.replace("Sir", "सर")
    return text

def detect_word_lang(word):
    devanagari_pattern = r'[\u0900-\u097F]'
    latin_pattern = r'[a-zA-Z]'
    if re.search(devanagari_pattern, word):
        return 'hi'
    elif re.search(latin_pattern, word):
        return 'en-us'
    else:
        return 'hi'  # Default to Hindi for symbols/unknown

def indian_english_post_process(phonemes):
    phonemes = re.sub(r'əʊ', 'oː', phonemes)
    phonemes = re.sub(r'eɪ', 'eː', phonemes)
    phonemes = re.sub(r'r\b', 'r', phonemes)
    return phonemes

def clean_phonemize(text):
    text = preprocess_text(text)
    # Split text to words and punctuation
    tokens = re.findall(r'[a-zA-Z\u0900-\u097F]+|[।.!?,]', text, re.UNICODE)
    phoneme_list = []
    separator = Separator(phone="", word="", syllable="")
    for word in tokens:
        lang = detect_word_lang(word)
        # For punctuation, just append as is
        if lang not in ('hi', 'en-us'):
            phoneme_list.append(word)
            continue
        # Phonemize each word separately
        try:
            ph = phonemize(
                [word],
                language=lang,
                backend="espeak",
                separator=separator,
                strip=True,
                preserve_punctuation=True,
                njobs=1,
                language_switch="remove-flags"
            )[0]
            if lang == 'en-us':
                ph = indian_english_post_process(ph)
        except Exception as e:
            ph = word  # Fallback: original
        phoneme_list.append(ph)

    # Your custom post-processing rules
    phoneme_list = ['sər' if p == 'sɜː' else p for p in phoneme_list]
    phoneme_list = ['wɛbsaɪt' if p == 'wɛb saɪd' else p for p in phoneme_list]
    ph_combined = ' '.join(phoneme_list)
    return ph_combined

# Example usage:
text = "बहुत अच्छा। This is a web side for Sir."
ph_string = clean_phonemize(text)
print(ph_string)


bʌhʊt ʌcʰcʰaː ðɪs ɪz eː wɛbsaɪt fɔːɹ sʌɾ .


In [67]:
# text = "CRM_SALES AI असिस्टेड कॉल लॉग Received."
text = ''' 
नमस्ते Sir

मैं Conn lee टीम से बोल रही हूँ

आपने recent ly हमारी web side visit की थी communication solution के लिए, right?

Conn lee एक smart platform है जहा  आप Whats App, Voice Call, Screen Sharing सब कुछ एक ही जगह पर use कर सकते हैं  और वो भी with Al-powered agent suppord.

सिर्फ एक app में आपका customer suppord, sales और collaboration manage हो सकता है।

'''
text_1 = "मैं website पर जाकर Sir से मिलूँगा"



In [68]:
import re
from indic_numtowords import num2words

# Precompile regex patterns
_CURRENCY_PATTERN = re.compile(r"(\d+)\s*Rs")
_YEAR_PATTERN = re.compile(r"\b(\d{4})\b")
_PHONE_SUB_PATTERN = re.compile(r"(\+?\d[\d\s\-]{6,}\d)")
_PHONE_CLEANUP_PATTERN = re.compile(r"^[\d\s\-+]+$")


def _replace_currency(match: re.Match) -> str:
    amount = int(match.group(1))
    hindi_amount = num2words(amount, lang='hi')
    return f"{hindi_amount} रुपये"


def _replace_year(match: re.Match) -> str:
    year = int(match.group(1))
    return num2words(year, lang='hi')


def _normalize_phone_number(text: str) -> str:
    """
    Converts a string of digits (with optional separators) into Hindi spoken digits.
    Returns the spoken form, or the original text if it doesn't match a phone pattern.
    """
    # Ensure the captured substring is a valid phone-like sequence
    if not _PHONE_CLEANUP_PATTERN.match(text) or sum(c.isdigit() for c in text) < 7:
        return text

    parts = []
    for ch in text:
        if ch.isdigit():
            parts.append(num2words(int(ch), lang='hi'))
        else:
            parts.append(ch)
    return " ".join(parts)


def normalize_text(text: str) -> str:
    """
    Normalizes phone numbers, currency (e.g., "1500 Rs"), and years (4-digit) in the input text to spoken Hindi.
    Processing steps:
      1. Inline phone-number substitution
      2. Currency normalization
      3. Year normalization
    """
    # 1. Replace all phone-like substrings
    def _phone_callback(m: re.Match) -> str:
        return _normalize_phone_number(m.group(1))
    text = _PHONE_SUB_PATTERN.sub(_phone_callback, text)

    # 2. Currency normalization
    text = _CURRENCY_PATTERN.sub(_replace_currency, text)

    # 3. Year normalization
    text = _YEAR_PATTERN.sub(_replace_year, text)

    return text


# Example usage
if __name__ == "__main__":
    sample = '''
    नमस्ते Sir

मैं Conn lee टीम से बोल रही हूँ

आपने recently हमारी web side visit की थी communication solution के लिए? शायद पिछले महीने, April 2025 के आस पास?

Conn lee एक smart platform है जहा आप Whats App, Voice Call, Screen Sharing सब कुछ एक ही जगह पर use कर सकते हैं और वो भी with Al-powered agent support.

अभी एक खास introductory offer चल रहा है, सिर्फ 1500 Rs प्रति माह पर! सिर्फ एक app में आपका customer support, sales और collaboration manage हो सकता है।

अगर आपके कोई सवाल हैं या आप डेमो देखना चाहते हैं, तो आप मुझे मेरे डायरेक्ट नंबर 9876543210 पर कॉल कर सकते हैं।
    '''
    text_2 = ''' 
Hello Team,

मुझे उम्मीद है कि आप सकुशल हैं। जैसा कि हमने तय किया था, अगला मीटिंग शेड्यूल 15,6,2025 को है, कृपया सुनिश्चित करें कि आपने 2500 Rs का एडवांस पेमेंट कर दिया हो,

यह नंबर 8012345678 पर कंफर्मेशन के लिए कॉल करें या 07551234567 पर व्हाट्सएप मैसेज भेजें,

धन्यवाद,
Project Coordinator

'''
    text_3 = ''' मैं website पर जाकर Sir से मिलूँगा'''
    # Normalize the text
    normalized_text = normalize_text(text_3)
    # Print the normalized text
    print(normalize_text(text_3))

 मैं website पर जाकर Sir से मिलूँगा


In [69]:
cleaned = clean_phonemize(text)
print(cleaned)
# print(word_tokenize(cleaned[0]))
# print(' '.join((cleaned)))

nəmʌsteː sʌɾ mɛ̃ kɑːn liː ʈiːm seː boːl ɾəhi hũ aːpneː ɹiːsənt laɪ həmaːɾi wɛbsaɪt vɪzɪt ki tʰi kəmjuːnɪkeːʃən səluːʃən keː lɪeː , ɹaɪt ? kɑːn liː eːk smɑːɹt plætfɔːɹm hɛː ɟʌhaː aːp wʌts æp , vɔɪs kɔːl , skɹiːn ʃɛɹɪŋ sʌb kʊcʰ eːk hi ɟʌɡəh pʌɾ juːs kʌɾ sʌkteː hɛ̃ ɔːɾ ʋoː bʰi wɪð æl paʊɚd eːdʒənt səpoːɹd . sɪɾpʰ eːk æp mẽː aːpkaː kʌstəmɚ səpoːɹd , seːlz ɔːɾ kəlæbɚɹeːʃən mænɪdʒ hoː sʌktaː hɛː


### Synthesize speech

In [70]:

def inference(text, ref_s, alpha = 0.3, beta = 0.7, diffusion_steps=5, embedding_scale=1):
    text = text.strip()
    # ps = global_phonemizer.phonemize([text])
    # normalized_text = normalize_text(text)
    ps = clean_phonemize(text)
    print(ps)
    # ps = word_tokenize(ps[0])
    # ps = ' '.join(ps)
    tokens = textclenaer(ps)
    print("before tokens", tokens)
    tokens.insert(0, 0)
    # tokens.append(0)
    # tokens = [1] + tokens + [2]
    print("after tokens", tokens)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2) 

        s_pred = sampler(noise = torch.randn((1, 256)).unsqueeze(1).to(device), 
                                          embedding=bert_dur,
                                          embedding_scale=embedding_scale,
                                            features=ref_s, # reference from the same speaker as the embedding
                                             num_steps=diffusion_steps).squeeze(1)


        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        ref = alpha * ref + (1 - alpha)  * ref_s[:, :128]
        s = beta * s + (1 - beta)  * ref_s[:, 128:]

        d = model.predictor.text_encoder(d_en, 
                                         s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)
        # if torch.isnan(pred_dur).any():
              #print("!!! NaN found in pred_dur !!!")
              # Decide how to handle
              # return None # Example: Stop processing

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, 
                                F0_pred, N_pred, ref.squeeze().unsqueeze(0))
        wav_out = out.squeeze().cpu().numpy() # weird pulse at the end of the model, need to be fixed later

    # Apply minimal post-processing to avoid muffled sound
    try:
        # Normalize the output (but not too aggressively)
        wav_out = wav_out / (np.max(np.abs(wav_out)) + 1e-7) * 0.9
        
        # Remove DC offset
        wav_out = wav_out - np.mean(wav_out)
        
        # Apply a very gentle high-pass filter just to remove sub-bass rumble
        b, a = scipy.signal.butter(2, 40/(24000/2), 'highpass')
        wav_out = scipy.signal.filtfilt(b, a, wav_out)
    except Exception as e:
        print(f"Warning: Error during audio post-processing: {e}")
    
        
    # return out.squeeze().cpu().numpy()[..., :-50] # weird pulse at the end of the model, need to be fixed later
    return wav_out[..., :-50] # weird pulse at the end of the model, need to be fixed later



#### Basic synthesis (5 diffusion steps, seen speakers)

In [71]:
text_en1 = ''' Thus did this humane and right minded father comfort his unhappy daughter, 
and her mother embracing her again, did all she could to soothe her feelings'''
# text = '''hello sir CRM SALES AI असिस्टेड कॉल लॉग Received.'''
text_1 = '''जब मैंने, सुबह की ठंडी हवा में अपने घर की बालकनी से सूरज की पहली किरणों को धरती पर बिखरते हुए देखा, तो मन एक, अनोखी ऊर्जा और शांति से भर गया, मानो प्रकृति ने स्वयं आकर मेरे दिन की सुंदर शुरुआत की हो'''
text_test= ''' 
नमस्ते Sir

मैं Conn lee टीम से बोल रही हूँ,

आपने recently हमारी web side visit की थी communication solution के लिए, शायद पिछले महीने, Aprilके आस पास.


अभी एक खास introductory offerr चल रहा है, सिर्फ fixfteen hundred Rupees प्रति माह पर,

अगर आपके कोई सवाल हैं या आप डेमो देखना चाहते हैं, तो आप मुझे मेरे डायरेक्ट नंबर पर कॉल कर सकते हैं।

'''
text_2 = '''
        Hello Team,

मुझे उम्मीद है कि आप सकुशल हैं। जैसा कि हमने तय किया था, अगली मीटिंग 15,6,2025 को निर्धारित है, इसलिए कृपया सुनिश्चित करें कि आपने 2500 Rs का एडवांस पेमेंट कर दिया है,

कृपया कंफर्मेशन के लिए 8012345678 पर कॉल करें, या व्हाट्सएप पर 07551234567 पर मैसेज भेजें,

धन्यवाद,
Project Coordinator
        '''
# मैं एक छोटा सा * demo schedule* करना चाहती हूँ।

# # क्या अभी 2 minute का time मिलेगा आपको ?
# text = "सुनो, ज़िंदगी छोटी है! tension कम लो, मुस्कान ज़्यादा दो. जो भी कर रहे हो, दिल से करो क्योंकि मेहनत कभी ज़ाया नहीं जाती।"
text_en = ''' 
you know you shouldn't give yourself a hard time for that, it's the best you can.
'''

In [72]:
# texts = {}
# texts['खुशी'] = "Paul's हम आपको अतीत की एक यात्रा पर आमंत्रित करते हुए प्रसन्न हैं, जहाँ हम मानव कृतियों द्वारा निर्मित सबसे अद्भुत स्मारकों का दर्शन करेंगे।"
# texts['Sad'] = "हमें यह बताते हुए खेद है कि हमारी समृद्धि और आत्मविश्वास को बहाल करने के प्रयासों में हमें गंभीर असफलता का सामना करना पड़ा है।"
# texts['Angry'] = "खगोलशास्त्र का क्षेत्र एक मज़ाक है! इसके सिद्धांत त्रुटिपूर्ण अवलोकनों और पक्षपाती व्याख्याओं पर आधारित हैं!"
# texts['Surprised'] = "मुझे विश्वास नहीं हो रहा! क्या आप सचमुच इस तालाब में बैक्टीरिया की एक नई प्रजाति की खोज कर चुके हैं?"


In [73]:
# for k, v in texts.items():
# 	# noise = torch.randn(1,1,256).to(device)
# 	ref_s = compute_style("/home/user/voice/StyleTTS2/Demo/reference/6.wav")
# 	wav = inference(v, ref_s, alpha=0.9, beta=0.7, diffusion_steps=20, embedding_scale=2)
# 	print(k + ": ")
# 	import IPython.display as ipd
# 	display(ipd.Audio(wav, rate=24000, normalize=False))

In [74]:
# import librosa
# import librosa.display
# import matplotlib.pyplot as plt
# import numpy as np
# import numpy as np
# import torch
# import librosa
# from scipy.stats import entropy
# from scipy.signal import stft
# def plot_spectogram(y, sr,to_mel=False, save_path=None, cmap='viridis'):
#     """
#     Plots and optionally saves the spectrogram from a raw audio signal (numpy array).

#     Parameters:
#     -----------
#     y : np.ndarray
#         The audio signal.
#     sr : int
#         The sampling rate of the audio signal.
#     to_mel : bool
#         If True, plots a Mel-spectrogram. Otherwise, plots a linear-frequency spectrogram.
#     save_path : str or None
#         If provided, saves the plot to this path.
#     cmap : str
#         Colormap for the spectrogram visualization.
#     """
#     if to_mel:
#         S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
#         S_db = librosa.power_to_db(S, ref=np.max)
#         y_axis = 'mel'
#         title = 'Mel Spectrogram (dB)'
#     else:
#         D = librosa.stft(y)
#         S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
#         y_axis = 'log'
#         title = 'Spectrogram (dB)'

#     # Plot
#     plt.figure(figsize=(4, 4))
#     librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis=y_axis, cmap=cmap)
#     plt.colorbar(format='%+2.0f dB')
#     plt.title(title)
#     plt.xlabel('Time (s)')
#     plt.ylabel('Frequency (Hz)' if not to_mel else 'Mel bins')
#     plt.tight_layout()

#     if save_path:
#         plt.savefig(save_path, dpi=300)
#         print(f"Spectrogram saved to {save_path}")
#     else:
#         plt.show()


In [75]:
def trim_audio(audio_np_array, sample_rate=24000, trim_ms=300):
    trim_samples = int(trim_ms * sample_rate / 1000)
    if len(audio_np_array) > 2 * trim_samples:
        trimmed_audio_np = audio_np_array[trim_samples:-trim_samples]
    else:
        trimmed_audio_np = audio_np_array
    return trimmed_audio_np

In [76]:
reference_dicts = {}
# reference_dicts['696_92939'] = "Demo/reference_audio/696_92939_000016_000006.wav"
# reference_dicts['1789_142896'] = "Demo/reference_audio/1789_142896_000022_000005.wav"
reference_dicts['Risha'] = "/home/user/voice/StyleTTS2/Demo/reference/risha.wav"
# reference_dicts["recorded"] = "/home/user/voice/data/styletts/audio/emotions/calm/wavs/Calm-1.wav"

In [77]:
# import time
# import numpy as np
# import torch
# import librosa
# from scipy.stats import entropy



# # your constants
# n_fft = 2048
# hop_length = 300
# win_length = 1200
# n_mels = 128
# eps = 1e-8
# sr_out = 24000

# start = time.time()

# audios = []
# for k, path in reference_dicts.items():
#     # --- generate synthesis ---
#     ref_s = compute_style(path)
#     old_wav = inference(text_en1, ref_s,
#                         alpha=0.5, beta=0.5,
#                         diffusion_steps=10,
#                         embedding_scale=1)

#     trimmed_audio = trim_audio(old_wav)    # your existing trim
#     audios.append(trimmed_audio)
#     wav = np.concatenate(audios)           # full synthesized sequence

#     # --- Load reference ---
#     y_true, sr_true = librosa.load(path, sr=None)
#     print("sr pred", sr_pred)
#     print("sr out", sr_out)
#     print("sr true", sr_true)
#     # --- Ensure same sampling rate for pred & true (resample if needed) ---
#     if sr_out != sr_true:
#         y_true = librosa.resample(y_true, orig_sr=sr_true, target_sr=sr_out)
#         sr_true = sr_out
#     y_pred = wav
#     sr_pred = sr_out
#     print("sr pred", sr_pred)
#     print("sr out", sr_out)
#     print("sr true", sr_true)
#     print(len(y_pred))
#     y_pred = y_pred[0:200000]
#     y_true = y_true[0:200000]
#     # --- 1) Mel-spectrograms with your hop/n_fft/win specs ---
#     S_true = librosa.feature.melspectrogram(
#         y=y_true,
#         sr=sr_true,
#         n_fft=n_fft,
#         hop_length=hop_length,
#         win_length=win_length,
#         n_mels=n_mels
#     )
#     S_pred = librosa.feature.melspectrogram(
#         y=y_pred,
#         sr=sr_pred,
#         n_fft=n_fft,
#         hop_length=hop_length,
#         win_length=win_length,
#         n_mels=n_mels
#     )

#     # --- 2) Truncate spectrograms to same #frames, then KL divergence ---
#     min_frames = min(S_true.shape[1], S_pred.shape[1])
#     print(S_true.shape[1], S_pred.shape[1])
#     S_true_cut = S_true[:, :min_frames]
#     S_pred_cut = S_pred[:, :min_frames]

#     p = (S_true_cut + eps) / np.sum(S_true_cut + eps)
#     q = (S_pred_cut + eps) / np.sum(S_pred_cut + eps)
#     kl_div = entropy(p.flatten(), q.flatten())

#     # --- 3) Align raw waveforms, then L1/L2 ---
#     min_len = min(len(y_true), len(y_pred))
#     print(min_len)
#     y_true_aligned = y_true[:min_len]
#     y_pred_aligned = y_pred[:min_len]

#     # as tensors: [batch=1, time]
#     y_true_t = torch.from_numpy(y_true_aligned).unsqueeze(0)
#     y_pred_t = torch.from_numpy(y_pred_aligned).unsqueeze(0)

#     l1_loss = torch.nn.functional.l1_loss(y_pred_t, y_true_t, reduction='mean')
#     l2_loss = torch.nn.functional.mse_loss(y_pred_t, y_true_t, reduction='mean')

#     # --- Print results ---
#     print(f"[{k}] KL Divergence (Mel): {kl_div:.6f}")
#     print(f"[{k}] L1 Loss (Waveform): {l1_loss.item():.6f}")
#     print(f"[{k}] L2 Loss (Waveform): {l2_loss.item():.6f}")

#     # --- Real-time factor ---
#     rtf = (time.time() - start) / (len(wav) / sr_out)
#     print(f"[{k}] RTF = {rtf:.5f}")

#     # --- Playback (optional) ---
#     import IPython.display as ipd
#     print(f"{k} Synthesized:")
#     display(ipd.Audio(wav, rate=sr_out, normalize=False))
#     plot_spectogram(y_pred, sr=24000)
#     print(f"{k} Reference:")
#     actual_wav, sr2 = librosa.load(path, sr = None)
#     display(ipd.Audio(path, rate=sr_out, normalize=False))
#     plot_spectogram(y_true, 24000)

#### With higher diffusion steps (more diverse)

Since the sampler is ancestral, the higher the stpes, the more diverse the samples are, with the cost of slower synthesis speed.

In [78]:
noise = torch.randn(1,1,256).to(device)
for k, path in reference_dicts.items():
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text_test, ref_s, alpha=0.3, beta=0.7, diffusion_steps=10, embedding_scale=1.5)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized:')
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print(k + ' Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))

nəmʌsteː sʌɾ mɛ̃ kɑːn liː ʈiːm seː boːl ɾəhi hũ , aːpneː ɹiːsəntli həmaːɾi wɛbsaɪt vɪzɪt ki tʰi kəmjuːnɪkeːʃən səluːʃən keː lɪeː , ʃaːjəd pɪcʰleː məhiːneː , eɪpɹəlkeː aːs paːs . ʌbʰi eːk kʰaːs ɪntɹədʌktɚɹi ɔfɛɹ cʌl ɾəhaː hɛː , sɪɾpʰ fɪksftiːn hʌndɹɪd ɹuːpiːz pɾʌtɪ maːh pʌɾ , ʌɡəɾ aːpkeː koːi səʋaːl hɛ̃ jaː aːp ɖeːmoː deːkʰnaː caːhəteː hɛ̃ , toː aːp mʊɟʰeː meːɾeː ɖaːjɾeːkʈ nʌmbəɾ pʌɾ kɔl kʌɾ sʌkteː hɛ̃
nəmʌsteː sʌɾ mɛ̃ kɑːn liː ʈiːm seː boːl ɾəhi hũ , aːpneː ɹiːsəntli həmaːɾi wɛbsaɪt vɪzɪt ki tʰi kəmjuːnɪkeːʃən səluːʃən keː lɪeː , ʃaːjəd pɪcʰleː məhiːneː , eɪpɹəlkeː aːs paːs . ʌbʰi eːk kʰaːs ɪntɹədʌktɚɹi ɔfɛɹ cʌl ɾəhaː hɛː , sɪɾpʰ fɪksftiːn hʌndɹɪd ɹuːpiːz pɾʌtɪ maːh pʌɾ , ʌɡəɾ aːpkeː koːi səʋaːl hɛ̃ jaː aːp ɖeːmoː deːkʰnaː caːhəteː hɛ̃ , toː aːp mʊɟʰeː meːɾeː ɖaːjɾeːkʈ nʌmbəɾ pʌɾ kɔl kʌɾ sʌkteː hɛ̃
nəmʌsteː sʌɾ mɛ̃ kɑːn liː ʈiːm seː boːl ɾəhi hũ , aːpneː ɹiːsəntli həmaːɾi wɛbsaɪt vɪzɪt ki tʰi kəmjuːnɪkeːʃən səluːʃən keː lɪeː , ʃaːjəd pɪcʰleː məhiːneː , eɪpɹəlkeː aːs paːs . ʌbʰi eːk 

Risha Reference:


#### Basic synthesis (5 diffusion steps, umseen speakers)
The following samples are to reproduce samples in [Section 4](https://styletts2.github.io/#libri) of the demo page. All spsakers are unseen during training. You can compare the generated samples to popular zero-shot TTS models like Vall-E and NaturalSpeech 2.

In [ ]:
reference_dicts = {}
# format: (path, text)
reference_dicts['1221-135767'] = ("Demo/reference_audio/1221-135767-0014.wav", "Yea, his honourable worship is within, but he hath a godly minister or two with him, and likewise a leech.")
reference_dicts['5639-40744'] = ("Demo/reference_audio/5639-40744-0020.wav", "Thus did this humane and right minded father comfort his unhappy daughter, and her mother embracing her again, did all she could to soothe her feelings.")
reference_dicts['908-157963'] = ("Demo/reference_audio/908-157963-0027.wav", "And lay me down in my cold bed and leave my shining lot.")
reference_dicts['4077-13754'] = ("Demo/reference_audio/4077-13754-0000.wav", "The army found the people in poverty and left them in comparative wealth.")

In [ ]:
noise = torch.randn(1,1,256).to(device)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized: ' + text)
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print(k + ' Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))

### Speech expressiveness

The following section recreates the samples shown in [Section 6](https://styletts2.github.io/#emo) of the demo page. The speaker reference used is `1221-135767-0014.wav`, which is unseen during training. 

#### With `embedding_scale=1`
This is the classifier-free guidance scale. The higher the scale, the more conditional the style is to the input text and hence more emotional.



In [24]:
ref_s = compute_style("/home/user/voice/data/styletts/audio/emotions/calm/wavs/Calm-4.wav")

In [29]:
texts = {}
texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"
# audios = []
for k,v in texts.items():
    audios = []
    old_wav = inference(v, ref_s, diffusion_steps=10, alpha=0.3, beta=0.9, embedding_scale=2)
    # old_wav = inference(text_en, ref_s, alpha=0, beta=0, diffusion_steps=10, embedding_scale=1.5)

    trimmed_audio = trim_audio(old_wav)
    audios.append(trimmed_audio)

    wav = np.concatenate(audios)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))
    # plot_spectogram(wav, 24000, to_mel=True)

['wiː ɑːɹ hæpi tʊ ɪnvaɪt juː tə dʒɔɪn ʌs ɔn ɐ dʒɜːni tə ðə pæst, wɛɹ wiː wɪl vɪzɪt ðə moʊst ɐmeɪzɪŋ mɑːnjuːmənts ɛvɚ bɪlt baɪ hjuːmən hændz.']
before tokens [65, 51, 158, 16, 69, 158, 123, 16, 50, 72, 58, 51, 16, 62, 135, 16, 102, 56, 64, 43, 102, 62, 16, 52, 63, 158, 16, 62, 83, 16, 46, 147, 76, 102, 56, 16, 138, 61, 16, 76, 56, 16, 70, 16, 46, 147, 87, 158, 56, 51, 16, 62, 83, 16, 81, 83, 16, 58, 72, 61, 62, 16, 3, 16, 65, 86, 123, 16, 65, 51, 158, 16, 65, 102, 54, 16, 64, 102, 68, 102, 62, 16, 81, 83, 16, 55, 57, 135, 61, 62, 16, 70, 55, 47, 102, 68, 102, 112, 16, 55, 69, 158, 56, 52, 63, 158, 55, 83, 56, 62, 61, 16, 86, 64, 85, 16, 44, 102, 54, 62, 16, 44, 43, 102, 16, 50, 52, 63, 158, 55, 83, 56, 16, 50, 72, 56, 46, 68, 16, 4]
after tokens [0, 65, 51, 158, 16, 69, 158, 123, 16, 50, 72, 58, 51, 16, 62, 135, 16, 102, 56, 64, 43, 102, 62, 16, 52, 63, 158, 16, 62, 83, 16, 46, 147, 76, 102, 56, 16, 138, 61, 16, 76, 56, 16, 70, 16, 46, 147, 87, 158, 56, 51, 16, 62, 83, 16, 81, 83, 16, 5

['aɪɐm sɑːɹi tə seɪ ðæt wiː hæv sʌfɚd ɐ sᵻvɪɹ sɛtbæk ɪn aʊɚɹ ɛfɚts tə ɹᵻstoːɹ pɹəspɛɹᵻɾi ænd kɑːnfɪdəns.']
before tokens [43, 102, 70, 55, 16, 61, 69, 158, 123, 51, 16, 62, 83, 16, 61, 47, 102, 16, 81, 72, 62, 16, 65, 51, 158, 16, 50, 72, 64, 16, 61, 138, 48, 85, 46, 16, 70, 16, 61, 177, 64, 102, 123, 16, 61, 86, 62, 44, 72, 53, 16, 102, 56, 16, 43, 135, 85, 123, 16, 86, 48, 85, 62, 61, 16, 62, 83, 16, 123, 177, 61, 62, 57, 158, 123, 16, 58, 123, 83, 61, 58, 86, 123, 177, 125, 51, 16, 72, 56, 46, 16, 53, 69, 158, 56, 48, 102, 46, 83, 56, 61, 16, 4]
after tokens [0, 43, 102, 70, 55, 16, 61, 69, 158, 123, 51, 16, 62, 83, 16, 61, 47, 102, 16, 81, 72, 62, 16, 65, 51, 158, 16, 50, 72, 64, 16, 61, 138, 48, 85, 46, 16, 70, 16, 61, 177, 64, 102, 123, 16, 61, 86, 62, 44, 72, 53, 16, 102, 56, 16, 43, 135, 85, 123, 16, 86, 48, 85, 62, 61, 16, 62, 83, 16, 123, 177, 61, 62, 57, 158, 123, 16, 58, 123, 83, 61, 58, 86, 123, 177, 125, 51, 16, 72, 56, 46, 16, 53, 69, 158, 56, 48, 102, 46, 83, 56, 61, 16

['ðə fiːld ʌv ɐstɹɑːnəmi ɪz ɐ dʒoʊk! ɪts θiəɹiz ɑːɹ beɪst ɔn flɔːd ɑːbzɚveɪʃənz ænd baɪəst ɪntɜːpɹɪteɪʃənz!']
before tokens [81, 83, 16, 48, 51, 158, 54, 46, 16, 138, 64, 16, 70, 61, 62, 123, 69, 158, 56, 83, 55, 51, 16, 102, 68, 16, 70, 16, 46, 147, 57, 135, 53, 16, 5, 16, 102, 62, 61, 16, 119, 51, 83, 123, 51, 68, 16, 69, 158, 123, 16, 44, 47, 102, 61, 62, 16, 76, 56, 16, 48, 54, 76, 158, 46, 16, 69, 158, 44, 68, 85, 64, 47, 102, 131, 83, 56, 68, 16, 72, 56, 46, 16, 44, 43, 102, 83, 61, 62, 16, 102, 56, 62, 87, 158, 58, 123, 102, 62, 47, 102, 131, 83, 56, 68, 16, 5]
after tokens [0, 81, 83, 16, 48, 51, 158, 54, 46, 16, 138, 64, 16, 70, 61, 62, 123, 69, 158, 56, 83, 55, 51, 16, 102, 68, 16, 70, 16, 46, 147, 57, 135, 53, 16, 5, 16, 102, 62, 61, 16, 119, 51, 83, 123, 51, 68, 16, 69, 158, 123, 16, 44, 47, 102, 61, 62, 16, 76, 56, 16, 48, 54, 76, 158, 46, 16, 69, 158, 44, 68, 85, 64, 47, 102, 131, 83, 56, 68, 16, 72, 56, 46, 16, 44, 43, 102, 83, 61, 62, 16, 102, 56, 62, 87, 158, 58, 123, 

['aɪ kænt bᵻliːv ɪt! juː miːn tə tɛl miː ðæt juː hæv dɪskʌvɚd ɐ nuː spiːsiːz ʌv bæktɪɹiə ɪn ðɪs pɑːnd?']
before tokens [43, 102, 16, 53, 72, 56, 62, 16, 44, 177, 54, 51, 158, 64, 16, 102, 62, 16, 5, 16, 52, 63, 158, 16, 55, 51, 158, 56, 16, 62, 83, 16, 62, 86, 54, 16, 55, 51, 158, 16, 81, 72, 62, 16, 52, 63, 158, 16, 50, 72, 64, 16, 46, 102, 61, 53, 138, 64, 85, 46, 16, 70, 16, 56, 63, 158, 16, 61, 58, 51, 158, 61, 51, 158, 68, 16, 138, 64, 16, 44, 72, 53, 62, 102, 123, 51, 83, 16, 102, 56, 16, 81, 102, 61, 16, 58, 69, 158, 56, 46, 16, 6]
after tokens [0, 43, 102, 16, 53, 72, 56, 62, 16, 44, 177, 54, 51, 158, 64, 16, 102, 62, 16, 5, 16, 52, 63, 158, 16, 55, 51, 158, 56, 16, 62, 83, 16, 62, 86, 54, 16, 55, 51, 158, 16, 81, 72, 62, 16, 52, 63, 158, 16, 50, 72, 64, 16, 46, 102, 61, 53, 138, 64, 85, 46, 16, 70, 16, 56, 63, 158, 16, 61, 58, 51, 158, 61, 51, 158, 68, 16, 138, 64, 16, 44, 72, 53, 62, 102, 123, 51, 83, 16, 102, 56, 16, 81, 102, 61, 16, 58, 69, 158, 56, 46, 16, 6]
Surprised: 


#### With `embedding_scale=2`

In [ ]:
texts = {}
texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

for k,v in texts.items():
    noise = torch.randn(1,1,256).to(device)
    wav = inference(v, ref_s, diffusion_steps=10, alpha=0.3, beta=0.7, embedding_scale=2)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### With `embedding_scale=2, alpha = 0.5, beta = 0.9`
`alpha` and `beta` is the factor to determine much we use the style sampled based on the text instead of the reference. The higher the value of `alpha` and `beta`, the more suitable the style it is to the text but less similar to the reference. Using higher beta makes the synthesized speech more emotional, at the cost of lower similarity to the reference. `alpha` determines the timbre of the speaker while `beta` determines the prosody. 

In [ ]:
texts = {}
texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

for k,v in texts.items():
    noise = torch.randn(1,1,256).to(device)
    wav = inference(v, ref_s, diffusion_steps=10, alpha=0.5, beta=0.9, embedding_scale=2)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

### Zero-shot speaker adaptation
This section recreates the "Acoustic Environment Maintenance" and "Speaker’s Emotion Maintenance" demo in [Section 4](https://styletts2.github.io/#libri) of the demo page. You can compare the generated samples to popular zero-shot TTS models like Vall-E. Note that the model was trained only on LibriTTS, which is about 250 times fewer data compared to those used to trian Vall-E with similar or better effect for these maintainance. 

#### Acoustic Environment Maintenance

Since we want to maintain the acoustic environment in the speaker (timbre), we set  `alpha = 0` to make the speaker as closer to the reference as possible while only changing the prosody according to the text.  

In [ ]:
reference_dicts = {}
# format: (path, text)
reference_dicts['3'] = ("Demo/reference_audio/3.wav", "As friends thing I definitely I've got more male friends.")
reference_dicts['4'] = ("Demo/reference_audio/4.wav", "Everything is run by computer but you got to know how to think before you can do a computer.")
reference_dicts['5'] = ("Demo/reference_audio/5.wav", "Then out in LA you guys got a whole another ball game within California to worry about.")

In [ ]:
noise = torch.randn(1,1,256).to(device)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.0, beta=0.5, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print('Synthesized: ' + text)
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print('Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))

#### Speaker’s Emotion Maintenance

Since we want to maintain the emotion in the speaker (prosody), we set  `beta = 0.1` to make the speaker as closer to the reference as possible while having some diversity thruogh the slight timbre change.

In [30]:
reference_dicts = {}
# format: (path, text)
reference_dicts['Anger'] = ("/home/user/voice/data/styletts/audio/emotions/angry/wavs/Anger-1.wav", "We have to reduce the number of plastic bags.")
reference_dicts['Sleepy'] = ("/home/user/voice/data/styletts/audio/emotions/fear/wavs/Fear-1.wav", "We have to reduce the number of plastic bags.")
reference_dicts['Amused'] = ("/home/user/voice/data/styletts/audio/emotions/excited/wavs/Excited-1.wav", "We have to reduce the number of plastic bags.")
reference_dicts['Disgusted'] = ("/home/user/voice/data/styletts/audio/emotions/base/wavs/Base-1.wav", "We have to reduce the number of plastic bags.")

In [31]:
noise = torch.randn(1,1,256).to(device)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.3, beta=0.1, diffusion_steps=10, embedding_scale=1.5)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized: ' + text)
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print(k + ' Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))

['wiː hæv tə ɹᵻduːs ðə nʌmbɚɹ ʌv plæstɪk bæɡz.']
before tokens [65, 51, 158, 16, 50, 72, 64, 16, 62, 83, 16, 123, 177, 46, 63, 158, 61, 16, 81, 83, 16, 56, 138, 55, 44, 85, 123, 16, 138, 64, 16, 58, 54, 72, 61, 62, 102, 53, 16, 44, 72, 92, 68, 16, 4]
after tokens [0, 65, 51, 158, 16, 50, 72, 64, 16, 62, 83, 16, 123, 177, 46, 63, 158, 61, 16, 81, 83, 16, 56, 138, 55, 44, 85, 123, 16, 138, 64, 16, 58, 54, 72, 61, 62, 102, 53, 16, 44, 72, 92, 68, 16, 4]
RTF = 0.106837
Anger Synthesized: We have to reduce the number of plastic bags.


Anger Reference:


['wiː hæv tə ɹᵻduːs ðə nʌmbɚɹ ʌv plæstɪk bæɡz.']
before tokens [65, 51, 158, 16, 50, 72, 64, 16, 62, 83, 16, 123, 177, 46, 63, 158, 61, 16, 81, 83, 16, 56, 138, 55, 44, 85, 123, 16, 138, 64, 16, 58, 54, 72, 61, 62, 102, 53, 16, 44, 72, 92, 68, 16, 4]
after tokens [0, 65, 51, 158, 16, 50, 72, 64, 16, 62, 83, 16, 123, 177, 46, 63, 158, 61, 16, 81, 83, 16, 56, 138, 55, 44, 85, 123, 16, 138, 64, 16, 58, 54, 72, 61, 62, 102, 53, 16, 44, 72, 92, 68, 16, 4]
RTF = 0.098665
Sleepy Synthesized: We have to reduce the number of plastic bags.


Sleepy Reference:


['wiː hæv tə ɹᵻduːs ðə nʌmbɚɹ ʌv plæstɪk bæɡz.']
before tokens [65, 51, 158, 16, 50, 72, 64, 16, 62, 83, 16, 123, 177, 46, 63, 158, 61, 16, 81, 83, 16, 56, 138, 55, 44, 85, 123, 16, 138, 64, 16, 58, 54, 72, 61, 62, 102, 53, 16, 44, 72, 92, 68, 16, 4]
after tokens [0, 65, 51, 158, 16, 50, 72, 64, 16, 62, 83, 16, 123, 177, 46, 63, 158, 61, 16, 81, 83, 16, 56, 138, 55, 44, 85, 123, 16, 138, 64, 16, 58, 54, 72, 61, 62, 102, 53, 16, 44, 72, 92, 68, 16, 4]
RTF = 0.096072
Amused Synthesized: We have to reduce the number of plastic bags.


Amused Reference:


['wiː hæv tə ɹᵻduːs ðə nʌmbɚɹ ʌv plæstɪk bæɡz.']
before tokens [65, 51, 158, 16, 50, 72, 64, 16, 62, 83, 16, 123, 177, 46, 63, 158, 61, 16, 81, 83, 16, 56, 138, 55, 44, 85, 123, 16, 138, 64, 16, 58, 54, 72, 61, 62, 102, 53, 16, 44, 72, 92, 68, 16, 4]
after tokens [0, 65, 51, 158, 16, 50, 72, 64, 16, 62, 83, 16, 123, 177, 46, 63, 158, 61, 16, 81, 83, 16, 56, 138, 55, 44, 85, 123, 16, 138, 64, 16, 58, 54, 72, 61, 62, 102, 53, 16, 44, 72, 92, 68, 16, 4]
RTF = 0.083268
Disgusted Synthesized: We have to reduce the number of plastic bags.


Disgusted Reference:


### Longform Narration

This section includes basic implementation of Algorithm 1 in the paper for consistent longform audio generation. The example passage is taken from [Section 5](https://styletts2.github.io/#long) of the demo page.

In [ ]:
passage = '''If the supply of fruit is greater than the family needs, it may be made a source of income by sending the fresh fruit to the market if there is one near enough, or by preserving, canning, and making jelly for sale. To make such an enterprise a success the fruit and work must be first class. There is magic in the word "Homemade," when the product appeals to the eye and the palate; but many careless and incompetent people have found to their sorrow that this word has not magic enough to float inferior goods on the market. As a rule large canning and preserving establishments are clean and have the best appliances, and they employ chemists and skilled labor. The home product must be very good to compete with the attractive goods that are sent out from such establishments. Yet for first class home made products there is a market in all large cities. All first-class grocers have customers who purchase such goods.'''

In [ ]:
def LFinference(text, s_prev, ref_s, alpha = 0.3, beta = 0.7, t = 0.7, diffusion_steps=5, embedding_scale=1):
    text = text.strip()
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)
    ps = ps.replace('``', '"')
    ps = ps.replace("''", '"')

    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2) 

        s_pred = sampler(noise = torch.randn((1, 256)).unsqueeze(1).to(device), 
                                          embedding=bert_dur,
                                          embedding_scale=embedding_scale,
                                            features=ref_s, # reference from the same speaker as the embedding
                                             num_steps=diffusion_steps).squeeze(1)
        
        if s_prev is not None:
            # convex combination of previous and current style
            s_pred = t * s_prev + (1 - t) * s_pred
        
        s = s_pred[:, 128:]
        ref = s_pred[:, :128]
        
        ref = alpha * ref + (1 - alpha)  * ref_s[:, :128]
        s = beta * s + (1 - beta)  * ref_s[:, 128:]

        s_pred = torch.cat([ref, s], dim=-1)

        d = model.predictor.text_encoder(d_en, 
                                         s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)


        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, 
                                F0_pred, N_pred, ref.squeeze().unsqueeze(0))
    
        
    return out.squeeze().cpu().numpy()[..., :-100], s_pred # weird pulse at the end of the model, need to be fixed later

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
s_ref = compute_style(path)
sentences = passage.split('.') # simple split by comma
wavs = []
s_prev = None
for text in sentences:
    if text.strip() == "": continue
    text += '.' # add it back
    
    wav, s_prev = LFinference(text, 
                              s_prev, 
                              s_ref, 
                              alpha = 0.3, 
                              beta = 0.9,  # make it more suitable for the text
                              t = 0.7, 
                              diffusion_steps=10, embedding_scale=1.5)
    wavs.append(wav)
print('Synthesized: ')
display(ipd.Audio(np.concatenate(wavs), rate=24000, normalize=False))
print('Reference: ')
display(ipd.Audio(path, rate=24000, normalize=False))

### Style Transfer

The following section demostrates the style transfer capacity for unseen speakers in [Section 6](https://styletts2.github.io/#emo) of the demo page. For this, we set `alpha=0.5, beta = 0.9` for the most pronounced effects (mostly using the sampled style). 

In [ ]:
def STinference(text, ref_s, ref_text, alpha = 0.3, beta = 0.7, diffusion_steps=5, embedding_scale=1):
    text = text.strip()
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)

    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    ref_text = ref_text.strip()
    ps = global_phonemizer.phonemize([ref_text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)

    ref_tokens = textclenaer(ps)
    ref_tokens.insert(0, 0)
    ref_tokens = torch.LongTensor(ref_tokens).to(device).unsqueeze(0)
    
    
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2) 
        
        ref_input_lengths = torch.LongTensor([ref_tokens.shape[-1]]).to(device)
        ref_text_mask = length_to_mask(ref_input_lengths).to(device)
        ref_bert_dur = model.bert(ref_tokens, attention_mask=(~ref_text_mask).int())
        s_pred = sampler(noise = torch.randn((1, 256)).unsqueeze(1).to(device), 
                                          embedding=bert_dur,
                                          embedding_scale=embedding_scale,
                                            features=ref_s, # reference from the same speaker as the embedding
                                             num_steps=diffusion_steps).squeeze(1)


        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        ref = alpha * ref + (1 - alpha)  * ref_s[:, :128]
        s = beta * s + (1 - beta)  * ref_s[:, 128:]

        d = model.predictor.text_encoder(d_en, 
                                         s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)


        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, 
                                F0_pred, N_pred, ref.squeeze().unsqueeze(0))
    
        
    return out.squeeze().cpu().numpy()[..., :-50] # weird pulse at the end of the model, need to be fixed later

In [ ]:
# reference texts to sample styles

ref_texts = {}
ref_texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
ref_texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
ref_texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
ref_texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

In [ ]:
path = "Demo/reference_audio/1221-135767-0014.wav"
s_ref = compute_style(path)

text = "Yea, his honourable worship is within, but he hath a godly minister or two with him, and likewise a leech."
for k,v in ref_texts.items():
    wav = STinference(text, s_ref, v, diffusion_steps=10, alpha=0.5, beta=0.9, embedding_scale=1.5)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

### Speech diversity

This section reproduces samples in [Section 7](https://styletts2.github.io/#var) of the demo page. 

`alpha` and `beta` determine the diversity of the synthesized speech. There are two extreme cases:
- If `alpha = 1` and `beta = 1`, the synthesized speech sounds the most dissimilar to the reference speaker, but it is also the most diverse (each time you synthesize a speech it will be totally different). 
- If `alpha = 0` and `beta = 0`, the synthesized speech sounds the most siimlar to the reference speaker, but it is deterministic (i.e., the sampled style is not used for speech synthesis). 


#### Default setting (`alpha = 0.3, beta=0.7`)
This setting uses 70% of the reference timbre and 30% of the reference prosody and use the diffusion model to sample them based on the text. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0.3, beta=0.7, embedding_scale=1)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### Less diverse setting (`alpha = 0.1, beta=0.3`)
This setting uses 90% of the reference timbre and 70% of the reference prosody. This makes it more similar to the reference speaker at cost of less diverse samples. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0.1, beta=0.3, embedding_scale=1)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### More diverse setting (`alpha = 0.5, beta=0.95`)
This setting uses 50% of the reference timbre and 5% of the reference prosody (so it uses 100% of the sampled prosody, which makes it more diverse), but this makes it more dissimilar to the reference speaker.  

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0.5, beta=0.95, embedding_scale=1)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### Extreme setting (`alpha = 1, beta=1`)
This setting uses 0% of the reference timbre and prosody and use the diffusion model to sample the entire style. This makes the speaker very dissimilar to the reference speaker. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=1, beta=1, embedding_scale=1)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### No variation (`alpha = 0, beta=0`)
This setting uses 0% of the reference timbre and prosody and use the diffusion model to sample the entire style. This makes the speaker very similar to the reference speaker, but there is no variation. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0, beta=0, embedding_scale=1)
    display(ipd.Audio(wav, rate=24000, normalize=False))

### Extra fun!

Here we clone some of the authors' voice of the StyleTTS 2 papers with a few seconds of the recording in the wild. None of the voices is in the dataset and all authors agreed to have their voices cloned here.

In [ ]:
text = ''' StyleTTS 2 is a text to speech model that leverages style diffusion and adversarial training with large speech language models to achieve human level text to speech synthesis. '''

In [ ]:
reference_dicts = {}
reference_dicts['Yinghao'] = "Demo/reference_audio/Yinghao.wav"
reference_dicts['Gavin'] = "Demo/reference_audio/Gavin.wav"
reference_dicts['Vinay'] = "Demo/reference_audio/Vinay.wav"
reference_dicts['Nima'] = "Demo/reference_audio/Nima.wav"

In [ ]:
start = time.time()
noise = torch.randn(1,1,256).to(device)
for k, path in reference_dicts.items():
    ref_s = compute_style(path)
    
    wav = inference(text, ref_s, alpha=0.1, beta=0.5, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print('Speaker: ' + k)
    import IPython.display as ipd
    print('Synthesized:')
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print('Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))